In [5]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()

/Users/shantanutaro/Documents/GitHub/Agentic AI/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [12]:
PRODUCTS = {
    "wireless headphones": {"price": 5999,  "rating": 4.6, "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."},
    "smart watch":         {"price": 4999, "rating": 4.3, "description": "Tracks heart rate and sleep. 5-day battery, water-resistant."},
    "mechanical keyboard": {"price": 2999, "rating": 4.8, "description": "Tenkeyless, Cherry MX Brown switches, per-key RGB."},
    "laptop stand":        {"price": 499,  "rating": 4.5, "description": "Adjustable aluminium, fits 11-17 inch laptops, folds flat."},
}
REVIEWS = {
    "wireless headphones": {"reviews": 1262, "rating": 4.6},
    "smart watch":         {"reviews": 340,  "rating": 3.9},
    "mechanical keyboard": {"reviews": 67,   "rating": 4.8},
    "laptop stand":        {"reviews": 781,  "rating": 4.5},
}

@tool
def get_product(name: str) -> str:
    """Look up a product by name and return its price, rating, stock and description."""
    p = PRODUCTS.get(name.lower())
    if not p:
        return(f"Product not found. Available: {', '.join(PRODUCTS)}")
    return str(p)

@tool
def get_review(name: str) -> str:
    """Look up a product review by product name and return its name, number of reviews and its rating."""
    p = REVIEWS.get(name.lower())
    if not p:
        return(f"Review not available for this product.")
    return str(p)

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

agent = create_agent(
    llm,
    tools = [get_product, get_review],
    system_prompt = "You are a helpful product assistant for an online tech store. Return price in Indian Rupees (₹)",
)

In [13]:
def ask(question: str):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [15]:
ask("what is the price of headphones and review")

The price of wireless headphones is ₹5999. The product has a rating of 4.6 and has 1262 reviews.


In [16]:
ask("how are headphones")

The price of wireless headphones is ₹5999. It has a rating of 4.6 and a description of "Over-ear Bluetooth, 30-hr battery, active noise cancellation."


In [17]:
ask("review of it")

Here are the reviews for the available products:

1. Wireless Headphones - 1262 reviews, 4.6 rating
2. Smart Watch - 340 reviews, 3.9 rating
3. Mechanical Keyboard - 67 reviews, 4.8 rating
4. Laptop Stand - 781 reviews, 4.5 rating

Please let me know if you would like to know the price of any of these products.


In [20]:
from langgraph.checkpoint.memory import InMemorySaver
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

agent2 = create_agent(
    llm,
    tools=[get_product, get_review],
    system_prompt = "You are a helpful product assistant for an online tech store. Return price in Indian Rupees (₹)",
    checkpointer = InMemorySaver()
)

def ask2(question: str):
    config = {"configurable": {"thread_id": "user-session-1"}}
    result = agent2.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config
    )
    print(result["messages"][-1].content)

ask2("What is the price of laptop stand")

In [22]:
ask2("What is the price of laptop stand")

The price of the laptop stand is ₹499.


In [23]:
ask2("what about review")

The laptop stand has 781 reviews with an average rating of 4.5.
